# AXIOM Constitutional AI — TinyLlama Fine-Tune

Fine-tunes TinyLlama-1.1B-Chat on 180 AXIOM constitutional AI examples using QLoRA.

**Pipeline:** Upload data → Load model (4-bit) → LoRA adapters → Train → Merge → GGUF → Download

**Requirements:** Colab T4 GPU (free tier), ~10 minutes training time

**Source:** github.com/Orivael-Dev/axiom | Patent Pending ORVL-001-PROV

In [ ]:
# Cell 1: Verify GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected.\n"
        "Go to: Runtime > Change runtime type > T4 GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# Cell 2: Install dependencies
# bitsandbytes unpinned — Colab CUDA version changes frequently
!pip install -q transformers==4.46.0 peft==0.13.0 trl==0.12.0
!pip install -q bitsandbytes accelerate==1.0.0
!pip install -q datasets scipy sentencepiece protobuf

# Verify bitsandbytes can find CUDA
import bitsandbytes as bnb
import torch
print(f"bitsandbytes: {bnb.__version__}")
print(f"CUDA (torch): {torch.version.cuda}")
print(f"bnb CUDA setup: OK")

In [ ]:
# Cell 3: Upload training data
# Select autotrain_data/train.jsonl from your local machine
from google.colab import files
import json

print("Select autotrain_data/train.jsonl from your local machine...")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
with open(filename, "r", encoding="utf-8") as f:
    examples = [json.loads(line) for line in f if line.strip()]

print(f"\nLoaded {len(examples)} training examples")
print(f"First example preview:")
print(examples[0]["text"][:300] + "...")

In [ ]:
# Cell 4: Prepare HuggingFace Dataset — 90/10 train/eval split
from datasets import Dataset

dataset = Dataset.from_list(examples)
dataset = dataset.shuffle(seed=42)

split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train: {len(train_dataset)} examples")
print(f"Eval:  {len(eval_dataset)} examples")

# Token length check
lengths = [len(ex["text"]) for ex in examples]
print(f"\nChar lengths — min: {min(lengths)}, max: {max(lengths)}, median: {sorted(lengths)[len(lengths)//2]}")
print(f"Est tokens — total: {sum(lengths)//4:,}, avg: {sum(lengths)//len(lengths)//4}")

In [ ]:
# Cell 5: Load TinyLlama in 4-bit (QLoRA)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Cell 6: Configure LoRA adapters
# rank=16, alpha=32 — enough capacity for domain shift, won't overfit 180 examples
# Targets: all 4 attention projections, skip MLP (unnecessary at this scale)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 7: Training config
# 4 epochs, effective batch 8, cosine LR 2e-4, eval every 20 steps
# Est. ~5-10 minutes on T4
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="./axiom-tinyllama-lora",

    # Duration
    num_train_epochs=4,

    # Batch (effective = 1 * 8 = 8)
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=1,

    # Learning rate
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # Sequence
    max_seq_length=1024,

    # Precision (T4 supports fp16, not bf16)
    fp16=True,
    bf16=False,

    # Evaluation
    eval_strategy="steps",
    eval_steps=20,

    # Checkpoints
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Logging
    logging_steps=5,
    report_to="none",

    # Optimizer
    optim="paged_adamw_8bit",
    weight_decay=0.05,
    max_grad_norm=0.3,

    # Dataset
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

print(f"Training steps: {trainer.state.max_steps if hasattr(trainer.state, 'max_steps') else 'TBD'}")
print(f"Ready to train.")

In [ ]:
# Cell 8: Train
import time

t0 = time.time()
trainer.train()
elapsed = time.time() - t0

print(f"\nTraining complete in {elapsed/60:.1f} minutes")
print(f"Final train loss: {trainer.state.log_history[-1].get('train_loss', 'N/A')}")
print(f"Best eval loss:   {min(h.get('eval_loss', 999) for h in trainer.state.log_history if 'eval_loss' in h):.4f}")

In [ ]:
# Cell 9: Save LoRA adapters
trainer.save_model("./axiom-tinyllama-lora/final")
tokenizer.save_pretrained("./axiom-tinyllama-lora/final")
print("LoRA adapters saved to ./axiom-tinyllama-lora/final")

import os
adapter_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fns in os.walk("./axiom-tinyllama-lora/final")
    for f in fns
) / (1024 * 1024)
print(f"Adapter size: {adapter_size:.1f} MB")

In [ ]:
# Cell 10: Merge LoRA into base model
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Reload base in float16 for clean merge
base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16,
    device_map="auto",
)

# Merge LoRA weights into base
model = PeftModel.from_pretrained(base_model, "./axiom-tinyllama-lora/final")
model = model.merge_and_unload()

# Save merged model
model.save_pretrained("./axiom-tinyllama-merged")
tok = AutoTokenizer.from_pretrained("./axiom-tinyllama-lora/final")
tok.save_pretrained("./axiom-tinyllama-merged")

merged_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fns in os.walk("./axiom-tinyllama-merged")
    for f in fns
) / (1024**3)
print(f"Merged model saved: {merged_size:.2f} GB")

In [ ]:
# Cell 11: Convert to GGUF Q4_K_M for Ollama deployment
import subprocess, os

# Install GGUF conversion tools
!pip install -q gguf

# Clone llama.cpp (conversion scripts only)
if not os.path.exists("llama.cpp"):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp.git

# Step 1: HF → GGUF F16
print("Converting to GGUF F16...")
!python llama.cpp/convert_hf_to_gguf.py \
    ./axiom-tinyllama-merged \
    --outfile axiom-tinyllama-f16.gguf \
    --outtype f16

# Step 2: Build quantize tool
print("\nBuilding llama-quantize...")
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release && \
    cmake --build build --target llama-quantize -j$(nproc)

# Step 3: Quantize to Q4_K_M (~600-700 MB)
print("\nQuantizing to Q4_K_M...")
!./llama.cpp/build/bin/llama-quantize \
    axiom-tinyllama-f16.gguf \
    axiom-tinyllama-Q4_K_M.gguf \
    Q4_K_M

size_mb = os.path.getsize("axiom-tinyllama-Q4_K_M.gguf") / (1024 * 1024)
print(f"\nFinal: axiom-tinyllama-Q4_K_M.gguf ({size_mb:.0f} MB)")

In [ ]:
# Cell 12: Download GGUF — save to models/ in your repo
from google.colab import files

print("Downloading axiom-tinyllama-Q4_K_M.gguf...")
print("Save to: i:/vsCode/promt-agent/models/")
files.download("axiom-tinyllama-Q4_K_M.gguf")

print("\nDeployment:")
print("  cd models/")
print("  ollama create axiom-tinyllama -f Modelfile")
print("  ollama run axiom-tinyllama")